# 🗂️ AI Chief of Staff — GRPO Training

Train Qwen2.5-3B-Instruct using GRPO (Group Relative Policy Optimization) on the Chief of Staff environment.

**Recommended Setup:**
- Google Colab with T4 GPU (free tier)
- Or Colab Pro with A100 for faster training

**Training Time:**
- T4: ~2-3 hours for 3 epochs
- A100: ~30-45 minutes

**Expected Results:**
- Email: 0.36 → 0.75+
- Calendar: 0.75 → 0.90+
- Delegation: 0.15 → 0.80+

In [ ]:
# 1️⃣ Install dependencies
!pip install -q unsloth transformers trl accelerate peft bitsandbytes
!pip install -q fastapi uvicorn pydantic requests

In [ ]:
# 2️⃣ Clone the environment from HuggingFace
import os
import sys

# Replace with your HuggingFace username
HF_USERNAME = "YOUR_USERNAME"  # ⚠️ CHANGE THIS

if not os.path.exists('/content/cos-env'):
    !git clone https://huggingface.co/spaces/{HF_USERNAME}/ai-chief-of-staff /content/cos-env

sys.path.insert(0, '/content/cos-env')
print("✅ Environment cloned")

In [ ]:
# 3️⃣ Start the environment server in background
import subprocess
import time
import requests

# Kill any existing server
!pkill -f "uvicorn server.app:app"
time.sleep(1)

# Start new server
server_process = subprocess.Popen(
    ['python3', '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '7860'],
    cwd='/content/cos-env',
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to start
for i in range(10):
    try:
        response = requests.get('http://localhost:7860/reset?task_id=easy_cos', timeout=2)
        if response.status_code == 200:
            print("✅ Server started successfully")
            break
    except:
        time.sleep(1)
else:
    print("⚠️ Server may not have started properly")

In [ ]:
# 4️⃣ Load model with Unsloth (4-bit quantization)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,  # Auto-detect
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
 defined')

In [ ]:
# Step 6 — Configure and run GRPO training
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir='/content/cos-grpo-output',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    logging_steps=10,
    save_steps=100,
    report_to='none',
)

# TODO: wire GRPOTrainer with run_episode_reward and a prompt dataset
# trainer = GRPOTrainer(
#     model=model,
#     args=training_args,
#     reward_funcs=[run_episode_reward],
#     train_dataset=prompt_dataset,
# )
# trainer.train()
print('Training config ready — fill in dataset and uncomment trainer to run')

In [ ]:
# Step 7 — Evaluate after training
# Run smoke test against trained model and compare to baseline
# Expected improvement: delegation 0.15 → 0.80+
print('After training, run: python3 test_smoke.py')
print('Then run: python3 plots/generate_chart.py to update the comparison chart')